# SwiftSQL V2: 批量并行评测 + 多轮迭代执行引导 DPO + 自动择优导出 GGUF

> **核心升级**：
> 1. **长度分桶 Batch 并行推理**：评测（`batch=16`）与候选采样（`batch=8 × 4/6 候选`）提速 **3～5 倍**；另附 20 题单并发实测，精准反映真实 API 延迟；
> 2. **多轮迭代执行引导 DPO（默认 2 轮，可改为 3 轮）**：每轮用最新进化的模型在真实 SQLite 沙箱里采样发现新错题，以逐轮衰减学习率 + 递增 KL 约束（`beta`）持续提分且不遗忘；
> 3. **自动择优导出**：对比 `Base vs V1 SFT vs V2-R1 vs V2-R2`，自动挑选综合表现最好的一版导出 `Q8_0` 和 `Q4_K_M` GGUF 到 Google Drive，并自动刷盘释放配额。


In [ ]:
# 1. 挂载 Google Drive、拉取最新代码并安装依赖
import os, sys, time, torch, yaml
from pathlib import Path
from google.colab import drive

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive", force_remount=True)
else:
    print("Google Drive already mounted at /content/drive")

DRIVE_DIR = "/content/drive/MyDrive/swift-sql"
REPO_DIR = "/content/swift-sql"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/2422725977/swift-sql.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

!pip install -q --upgrade unsloth trl datasets sqlglot pyyaml
!pip install -q --no-deps -e {REPO_DIR}
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

with open(f"{REPO_DIR}/configs/train_config.yaml", "r", encoding="utf-8") as f:
    CFG = yaml.safe_load(f)

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Ready: {name} ({vram:.1f} GB VRAM) | bf16={torch.cuda.is_bf16_supported()}")
else:
    raise RuntimeError("未检测到 GPU！请切换到 L4 或 A100 运行时。")


In [ ]:
# 2. 高速长度分桶批量推理引擎 + 统一评测函数
from tqdm.auto import tqdm
from unsloth import FastLanguageModel
from swiftsql.data_pipeline import load_jsonl, save_jsonl
from swiftsql.evaluation import evaluate_predictions

EVAL_PATH = Path(DRIVE_DIR) / "data" / "eval.jsonl"
TRAIN_PATH = Path(DRIVE_DIR) / "data" / "train_sft.jsonl"
eval_records = load_jsonl(EVAL_PATH)
train_records = load_jsonl(TRAIN_PATH)
dpo_cfg = CFG["dpo"]
print(f"Loaded eval={len(eval_records)}, train={len(train_records)}")

def _prepare_tokenizer_for_batch(tok):
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok

def batch_generate(
    model,
    tok,
    prompts,
    *,
    batch_size=16,
    max_new_tokens=256,
    do_sample=False,
    temperature=0.0,
    top_p=0.9,
    num_return_sequences=1,
    desc="Batched inference",
):
    """Length-bucketed left-padded batched generation for 3-5x speedup on L4/A100."""
    _prepare_tokenizer_for_batch(tok)
    # Sort prompts by char length to minimize padding waste, then restore original order
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    results = [None] * len(prompts)

    pbar = tqdm(range(0, len(order), batch_size), desc=desc)
    with torch.inference_mode():
        for start in pbar:
            idxs = order[start : start + batch_size]
            batch_prompts = [prompts[i] for i in idxs]
            inputs = tok(
                batch_prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=1536,
            ).to("cuda")
            in_len = inputs["input_ids"].shape[1]
            gen_kwargs = {
                "max_new_tokens": max_new_tokens,
                "do_sample": do_sample,
                "num_return_sequences": num_return_sequences,
                "pad_token_id": tok.pad_token_id,
                "eos_token_id": tok.eos_token_id,
            }
            if do_sample:
                gen_kwargs["temperature"] = temperature
                gen_kwargs["top_p"] = top_p
            outs = model.generate(**inputs, **gen_kwargs)
            decoded = tok.batch_decode(outs[:, in_len:], skip_special_tokens=True)
            if num_return_sequences == 1:
                for idx, txt in zip(idxs, decoded):
                    results[idx] = txt.strip()
            else:
                for pos, idx in enumerate(idxs):
                    chunk = decoded[pos * num_return_sequences : (pos + 1) * num_return_sequences]
                    results[idx] = [t.strip() for t in chunk]
    return results

def run_benchmark(model_name_or_path, records, max_samples=None, batch_size=None, label=None):
    max_samples = max_samples or dpo_cfg.get("eval_samples", 300)
    batch_size = batch_size or dpo_cfg.get("eval_batch_size", 16)
    subset = records[:max_samples]
    tag = label or Path(model_name_or_path).name

    m, tok = FastLanguageModel.from_pretrained(
        model_name=model_name_or_path,
        max_seq_length=CFG["model"]["max_seq_length"],
        dtype=None,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(m)
    prompts = [
        tok.apply_chat_template([x for x in r["messages"] if x["role"] != "assistant"], tokenize=False, add_generation_prompt=True)
        for r in subset
    ]

    t_all = time.perf_counter()
    preds = batch_generate(m, tok, prompts, batch_size=batch_size, do_sample=False, desc=f"Eval {tag}")
    total_s = time.perf_counter() - t_all

    # Measure true single-request latency on 20 representative samples
    single_lats = []
    with torch.inference_mode():
        for p in prompts[:20]:
            inp = tok([p], return_tensors="pt", truncation=True, max_length=1536).to("cuda")
            t0 = time.perf_counter()
            m.generate(**inp, max_new_tokens=256, do_sample=False, pad_token_id=tok.pad_token_id)
            single_lats.append((time.perf_counter() - t0) * 1000.0)

    report = evaluate_predictions(subset, preds, latencies_ms=single_lats)
    report.pop("details", None)
    report["throughput_qps"] = round(len(subset) / max(total_s, 1e-6), 2)
    del m
    torch.cuda.empty_cache()
    return report

REPORTS = {}
REPORTS["Base"] = run_benchmark(CFG["model"]["base_model"], eval_records, label="Base-3B")
REPORTS["V1_SFT"] = run_benchmark(f"{DRIVE_DIR}/models/v1_sft_lora", eval_records, label="V1-SFT")

def print_comparison(reports_dict):
    names = list(reports_dict.keys())
    header = f"{'Metric':22s} | " + " | ".join(f"{n:>10s}" for n in names)
    print("\n" + "=" * len(header))
    print(header)
    print("-" * len(header))
    keys = [
        "json_valid_rate",
        "syntax_valid_rate",
        "execution_accuracy",
        "refusal_f1",
        "avg_output_chars",
        "avg_latency_ms",
        "throughput_qps",
    ]
    for k in keys:
        row = f"{k:22s} | " + " | ".join(f"{str(reports_dict[n].get(k)):>10s}" for n in names)
        print(row)
    print("=" * len(header))

print_comparison(REPORTS)


In [ ]:
# 3. 多轮迭代执行引导 DPO（Iterative Execution-Guided DPO，默认 2 轮，可改为 3 轮）
import random
from datasets import Dataset
from unsloth import PatchDPOTrainer
from trl import DPOConfig, DPOTrainer
from swiftsql.dpo_builder import build_dpo_pair

PatchDPOTrainer()

DPO_ROUNDS = dpo_cfg.get("rounds", 2)  # 可改为 3 进一步打磨
NUM_PROMPTS_PER_ROUND = dpo_cfg.get("num_prompts_per_round", 1200)
CANDS_PER_ROUND = dpo_cfg.get("candidates_per_round", [4, 6])
LRS_PER_ROUND = dpo_cfg.get("round_learning_rates", [5e-5, 2.5e-5])
BETAS_PER_ROUND = dpo_cfg.get("round_betas", [0.10, 0.15])
SAMPLE_BATCH_SIZE = dpo_cfg.get("sample_batch_size", 8)
FORCE_RESAMPLE = False  # 设为 True 可忽略 Drive 缓存的偏好对重新采样

# 准备可执行题池（优先按复杂题比例分层）与拒答题池
rng = random.Random(CFG["project"]["seed"])
exec_hard = [r for r in train_records if r["meta"].get("executable_context") and r["complexity"] in ("medium", "complex")]
exec_easy = [r for r in train_records if r["meta"].get("executable_context") and r["complexity"] == "simple"]
refusal_all = [r for r in train_records if r["task_type"] == "refusal"]
rng.shuffle(exec_hard)
rng.shuffle(exec_easy)
rng.shuffle(refusal_all)
print(f"Pool ready: hard_exec={len(exec_hard)}, easy_exec={len(exec_easy)}, refusal={len(refusal_all)}")

current_model_path = f"{DRIVE_DIR}/models/v1_sft_lora"
MODEL_PATHS = {"V1_SFT": current_model_path}

for r_idx in range(1, DPO_ROUNDS + 1):
    round_tag = f"V2_R{r_idx}"
    n_cand = CANDS_PER_ROUND[min(r_idx - 1, len(CANDS_PER_ROUND) - 1)]
    lr = LRS_PER_ROUND[min(r_idx - 1, len(LRS_PER_ROUND) - 1)]
    beta = BETAS_PER_ROUND[min(r_idx - 1, len(BETAS_PER_ROUND) - 1)]
    pair_path = Path(DRIVE_DIR) / "data" / f"dpo_pairs_round{r_idx}.jsonl"
    out_lora_dir = f"{DRIVE_DIR}/models/v2_dpo_r{r_idx}_lora"

    print(f"\n================ Starting {round_tag} (cands={n_cand}, lr={lr}, beta={beta}) ================")

    # Step A: 按轮次切分新题池（第 2 轮起提升复杂题占比至 70% 专攻难例）
    hard_ratio = 0.55 if r_idx == 1 else 0.70
    n_hard = int(NUM_PROMPTS_PER_ROUND * hard_ratio)
    n_easy = NUM_PROMPTS_PER_ROUND - n_hard
    h_start = ((r_idx - 1) * n_hard) % max(len(exec_hard), 1)
    e_start = ((r_idx - 1) * n_easy) % max(len(exec_easy), 1)
    ref_start = ((r_idx - 1) * 150) % max(len(refusal_all), 1)

    pool = (
        (exec_hard[h_start:] + exec_hard[:h_start])[:n_hard]
        + (exec_easy[e_start:] + exec_easy[:e_start])[:n_easy]
        + (refusal_all[ref_start:] + refusal_all[:ref_start])[:150]
    )

    if pair_path.exists() and not FORCE_RESAMPLE:
        dpo_pairs = load_jsonl(pair_path)
        print(f"[{round_tag}] Loaded cached DPO pairs: {len(dpo_pairs)}")
    else:
        m, tok = FastLanguageModel.from_pretrained(
            model_name=current_model_path,
            max_seq_length=CFG["model"]["max_seq_length"],
            dtype=None,
            load_in_4bit=True,
        )
        FastLanguageModel.for_inference(m)
        prompts = [
            tok.apply_chat_template([x for x in rec["messages"] if x["role"] != "assistant"], tokenize=False, add_generation_prompt=True)
            for rec in pool
        ]
        # 显存自适应：候选数变多时微调 batch size 防止 OOM
        eff_bs = max(2, SAMPLE_BATCH_SIZE * 4 // n_cand)
        all_cands = batch_generate(
            m,
            tok,
            prompts,
            batch_size=eff_bs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.75,
            top_p=0.92,
            num_return_sequences=n_cand,
            desc=f"{round_tag} Sampling ({len(pool)}x{n_cand}, bs={eff_bs})",
        )
        dpo_pairs = []
        for rec, cands in zip(pool, all_cands):
            pair = build_dpo_pair(rec, cands)
            if pair is not None:
                dpo_pairs.append(pair)

        save_jsonl(dpo_pairs, pair_path)
        del m
        torch.cuda.empty_cache()
        print(f"[{round_tag}] Built {len(dpo_pairs)} execution-verified DPO pairs -> {pair_path}")

    if len(dpo_pairs) < 50:
        print(f"[{round_tag}] Too few preference pairs ({len(dpo_pairs)}), stopping early.")
        break

    # Step B: 90% 训练 + 10% 留出验证集监控 DPO loss（自动保留最优 checkpoint）
    rng.shuffle(dpo_pairs)
    val_size = max(20, min(150, int(len(dpo_pairs) * 0.10)))
    val_pairs = dpo_pairs[:val_size]
    trn_pairs = dpo_pairs[val_size:]

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=current_model_path,
        max_seq_length=CFG["model"]["max_seq_length"],
        dtype=None,
        load_in_4bit=True,
    )

    def _fmt(pairs):
        rows = []
        for p in pairs:
            rows.append({
                "prompt": tokenizer.apply_chat_template(p["prompt"], tokenize=False, add_generation_prompt=True),
                "chosen": p["chosen"][0]["content"] + tokenizer.eos_token,
                "rejected": p["rejected"][0]["content"] + tokenizer.eos_token,
            })
        return Dataset.from_list(rows)

    train_dpo_ds = _fmt(trn_pairs)
    eval_dpo_ds = _fmt(val_pairs)
    use_bf16 = torch.cuda.is_bf16_supported()

    trainer = DPOTrainer(
        model=model,
        ref_model=None,
        processing_class=tokenizer,
        train_dataset=train_dpo_ds,
        eval_dataset=eval_dpo_ds,
        args=DPOConfig(
            output_dir=f"{DRIVE_DIR}/checkpoints/v2_dpo_r{r_idx}",
            beta=beta,
            learning_rate=lr,
            num_train_epochs=dpo_cfg.get("num_train_epochs", 2),
            per_device_train_batch_size=dpo_cfg["per_device_train_batch_size"],
            gradient_accumulation_steps=dpo_cfg["gradient_accumulation_steps"],
            warmup_ratio=0.08,
            lr_scheduler_type="cosine",
            logging_steps=10,
            eval_strategy="steps",
            eval_steps=50,
            save_steps=50,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            save_total_limit=2,
            fp16=not use_bf16,
            bf16=use_bf16,
            optim="adamw_8bit",
            max_length=CFG["model"]["max_seq_length"],
            max_prompt_length=1536,
            report_to="none",
        ),
    )
    trainer.train()

    model.save_pretrained(out_lora_dir)
    tokenizer.save_pretrained(out_lora_dir)
    del model, trainer
    torch.cuda.empty_cache()

    # Step C: 立即在独立验证集上评测本轮进化结果
    REPORTS[round_tag] = run_benchmark(out_lora_dir, eval_records, label=round_tag)
    MODEL_PATHS[round_tag] = out_lora_dir
    current_model_path = out_lora_dir
    print_comparison(REPORTS)


In [ ]:
# 4. 自动择优（综合 EX + 语法率 + 拒答 F1）并导出最终 GGUF 到 Google Drive
def composite_score(rep):
    ex = rep.get("execution_accuracy") or 0.0
    syn = rep.get("syntax_valid_rate") or 0.0
    ref = rep.get("refusal_f1") or 0.0
    js = rep.get("json_valid_rate") or 0.0
    return 0.55 * ex + 0.20 * syn + 0.15 * ref + 0.10 * js

candidates = {k: v for k, v in REPORTS.items() if k in MODEL_PATHS}
best_tag = max(candidates.keys(), key=lambda k: composite_score(candidates[k]))
best_path = MODEL_PATHS[best_tag]

print_comparison(REPORTS)
print(f"\n>>> Best checkpoint selected: {best_tag} ({best_path}) | composite={composite_score(REPORTS[best_tag]):.4f}")

best_model, best_tok = FastLanguageModel.from_pretrained(
    model_name=best_path,
    max_seq_length=CFG["model"]["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

GGUF_V2_DIR = f"{DRIVE_DIR}/exports/v2_best_gguf"
os.makedirs(GGUF_V2_DIR, exist_ok=True)
for q in CFG["export"]["quantizations"]:
    print(f"Exporting {best_tag} GGUF ({q}) -> {GGUF_V2_DIR} ...")
    best_model.save_pretrained_gguf(GGUF_V2_DIR, best_tok, quantization_method=q)

!ls -lh {GGUF_V2_DIR}


In [ ]:
# 5. 释放 Colab Pro GPU 配额（先刷新 Google Drive 缓存，防止权重/GGUF 丢数据）
import os, time
from google.colab import drive, runtime

AUTO_UNASSIGN = True  # 如需保留当前会话继续调试，可改为 False

if AUTO_UNASSIGN:
    print("Syncing and flushing Google Drive before releasing GPU runtime...")
    os.sync()
    time.sleep(5)
    drive.flush_and_unmount()
    print("Google Drive flushed. Releasing Colab GPU runtime now to save compute units!")
    runtime.unassign()
else:
    print("AUTO_UNASSIGN=False: 保持当前会话。用完后请手动点击右上角「断开连接并删除运行时」以停止扣费。")
